# 02. Извлечение эмбеддингов XLM-R

Прогоняем все отзывы через `xlm-roberta-base` → получаем (N, 768) векторы.
Сохраняем `X_cls.npy`, `X_mean.npy`, `labels.npy`, `ratings.npy`, `meta.json`.


In [ ]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/USER/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath(".."))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)


In [ ]:
from embeddings.extract import extract_embeddings
import json, numpy as np
from pathlib import Path

INPUT = Path('../data/raw/reviews.jsonl')
OUT = Path('../data/embeddings')
OUT.mkdir(parents=True, exist_ok=True)

reviews = [json.loads(line) for line in INPUT.open(encoding='utf-8')]
print(f'reviews: {len(reviews)}')


In [ ]:
result = extract_embeddings(
    reviews,
    model_name='xlm-roberta-base',
    batch_size=32,
    max_len=128,
)

np.save(OUT / 'X_cls.npy', result['cls'])
np.save(OUT / 'X_mean.npy', result['mean'])
np.save(OUT / 'labels.npy', result['labels'])
np.save(OUT / 'ratings.npy', result['ratings'])

meta = {
    'cat_to_id': result['cat_to_id'],
    'n_reviews': len(reviews),
    'emb_dim': int(result['cls'].shape[1]),
}
(OUT / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2))
print('saved', meta)


## Проверка: казахский vs русский в эмбеддингах

Эмпирический тест из обсуждения — XLM-R понимает казахский?


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base').eval()

@torch.no_grad()
def emb(text):
    e = tok(text, return_tensors='pt', truncation=True)
    return model(**e).last_hidden_state[:, 0, :].numpy()

pairs = [
    ('хороший товар', 'жақсы өнім'),
    ('доставка быстрая', 'жеткізу жылдам'),
    ('качество ужасное', 'сапасы өте нашар'),
]
for ru, kk in pairs:
    sim = cosine_similarity(emb(ru), emb(kk))[0, 0]
    print(f'cos({ru!r}, {kk!r}) = {sim:.3f}')
